# Mouse embryo — CellSTIC tutorial

**Dataset:** Mouse embryo Stereo-seq; per-stage input under data/mouse_embryo/<stage>/raw/. See data/mouse_embryo/README.md.


## Step 1 — Setup


In [ ]:
import sys
import base64
from pathlib import Path

%load_ext autoreload
%autoreload 2

_cwd = Path.cwd()
project_root = _cwd if (_cwd / "model").is_dir() else _cwd.parent
sys.path.insert(0, str(project_root))

import anndata as ad
import numpy as np
import scanpy as sc
import scipy.sparse as sparse
import torch
from IPython.display import HTML, SVG, display
from scipy.spatial.distance import pdist, squareform

from model.train import CellSTICConfig
from pipeline import run_cellstic
from pipeline.analyzer import SingleLevelAnalysis
from utils.data import SpatialPreprocessorUtils
from utils.tools.celltypist_utils import CellTypistAnnotator, SPECIES_MODEL_MAP
from utils.tools.seed_utils import set_global_seed

set_global_seed()

def display_svg_grid(paths, n_cols=None, max_width=280):
    paths = [Path(p) for p in paths]
    if not paths:
        print("No SVG files found to display.")
        return
    if n_cols is None:
        n_cols = min(3, len(paths))
    rows = []
    for i in range(0, len(paths), n_cols):
        cells = []
        for path in paths[i : i + n_cols]:
            uri = "data:image/svg+xml;base64," + base64.b64encode(path.read_bytes()).decode("ascii")
            cells.append(
                f'<td style="width:{100 / n_cols:.2f}%;padding:4px;text-align:center">'
                f'<img src="{uri}" style="width:100%;max-width:{max_width}px;height:auto;"/>'
                f"</td>"
            )
        cells.extend(f'<td style="width:{100 / n_cols:.2f}%"></td>' for _ in range(n_cols - len(cells)))
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f'<table style="width:100%;table-layout:fixed;border-collapse:collapse"><tbody>{"".join(rows)}</tbody></table>'))

## Step 2 — Configuration


In [ ]:
cfg = CellSTICConfig()
cfg.model.graph.expression_percentile = 98
cfg.model.graph.n_spots = 4
cfg.model.tree.hierarchy_method = "balanced"
cfg.train.ccc.epochs = 50
cfg.train.ccc.learning_rate = 0.01
cfg.train.feat.epochs = 2000
cfg.train.feat.n_clusters = 7

STAGE = "14.5"
ORGAN = "Brain"  # "Brain" or "Liver"
USE_CACHE = True
ANNOTATION_MODEL_MAP = {"Brain": "mouse_brain", "Liver": "mouse_liver"}

work_dir = project_root / "data" / "mouse_embryo" / STAGE
raw_path = work_dir / "raw"
preprocess_path = work_dir / "preprocess" / ORGAN
model_path = work_dir / "model" / ORGAN
result_path = work_dir / "result" / ORGAN
analysis_path = work_dir / "analysis" / ORGAN

# dict[ligand -> list[receptor]]; edit as needed
ligand_receptor_map = {
    "F2": ["F2r"],
    "Lpar3": ["Adgre5"],
    "Nts": ["Sort1"],
    "Plg": ["Pard3"],
    "Thbs4": ["Cd36"],
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for path in [raw_path, preprocess_path, model_path, result_path, analysis_path]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Stage:            {STAGE}")
print(f"Organ:            {ORGAN}")
print(f"Device:           {device}")
print(f"Ligand entries:   {len(ligand_receptor_map)}")
print(f"Result path:      {result_path}")
print(f"Analysis path:    {analysis_path}")

## Step 3 — Load and Preprocess

Load the selected stage and organ, optionally reuse cached filtered AnnData, and build obsm['feat'] plus obsp['spatial_distances'] in memory.


In [ ]:
_CACHE_NAME_FILTERED = "preprocessed_RNA_filtered.h5ad"


def annotate_cell_type_by_domain(rna_adata, annotation_key, annotation_model_map, cell_type_key="cell_type"):
    if annotation_key not in rna_adata.obs:
        return rna_adata
    domains = rna_adata.obs[annotation_key].astype(str).unique()
    rna_adata.obs[cell_type_key] = ""
    for domain in domains:
        if domain not in annotation_model_map:
            continue
        model_spec = annotation_model_map[domain]
        mask = rna_adata.obs[annotation_key].astype(str) == domain
        sub = rna_adata[mask].copy()
        if sub.n_obs == 0:
            continue
        model_path = SPECIES_MODEL_MAP.get(model_spec, model_spec)
        annotator = CellTypistAnnotator(model=model_path, species=None)
        annotator.annotate(sub, output_path=None)
        rna_adata.obs.loc[mask, cell_type_key] = sub.obs["predicted_labels"].values

    labels = rna_adata.obs[cell_type_key]
    not_na = labels.notna()
    labels_str = labels[not_na].astype(str)
    has_colon = labels_str.str.contains(":", regex=False)
    if has_colon.any():
        labels_str.loc[has_colon] = labels_str.loc[has_colon].str.split(":", n=1).str[0].str.strip()
        rna_adata.obs.loc[not_na, cell_type_key] = labels_str.values
    rna_adata.obs[cell_type_key] = rna_adata.obs[cell_type_key].astype("category")
    print(f"Annotated {rna_adata.obs[cell_type_key].nunique()} cell types in obs[{cell_type_key!r}].")
    return rna_adata


def load_mouse_embryo_inline(
    raw_path,
    preprocess_path,
    use_cache=True,
    annotation_key="annotation",
    annotation_value=None,
    annotation_model_map=None,
):
    raw_path = Path(raw_path)
    preprocess_path = Path(preprocess_path)

    cache_path = preprocess_path / _CACHE_NAME_FILTERED
    if use_cache and cache_path.exists():
        return sc.read_h5ad(cache_path)

    raw_files = [path for path in raw_path.glob("*.h5ad") if path.name != _CACHE_NAME_FILTERED]
    if not raw_files:
        raise FileNotFoundError(f"No raw h5ad found under {raw_path.absolute()}")
    rna_adata = sc.read_h5ad(raw_files[0])

    if annotation_value is not None:
        if annotation_key not in rna_adata.obs:
            raise ValueError(f"Annotation key {annotation_key!r} not in adata.obs.")
        mask = rna_adata.obs[annotation_key].astype(str).eq(str(annotation_value)).values
        if mask.sum() == 0:
            available = sorted(rna_adata.obs[annotation_key].astype(str).unique())
            raise ValueError(f"No cells with annotation_value {annotation_value!r}. Available: {available}")
        before = rna_adata.n_obs
        rna_adata = rna_adata[mask, :].copy()
        print(f"Annotation filter {annotation_key}={annotation_value}: {before} -> {rna_adata.n_obs} cells")

    if "feat" in rna_adata.obsm and "spatial_distances" in rna_adata.obsp:
        if use_cache:
            preprocess_path.mkdir(parents=True, exist_ok=True)
            rna_adata.write_h5ad(cache_path)
        return rna_adata

    rna_adata.raw = rna_adata.copy()
    rna_adata.var["mt"] = rna_adata.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(rna_adata, percent_top=None, log1p=False, inplace=True)
    sc.pp.highly_variable_genes(rna_adata, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(rna_adata, target_sum=1e4)
    sc.pp.log1p(rna_adata)
    rna_adata.obsm["feat"] = SpatialPreprocessorUtils.pca(
        rna_adata[:, rna_adata.var["highly_variable"]], n_comps=500
    )
    rna_adata.obsp["spatial_distances"] = sparse.csr_matrix(
        squareform(pdist(rna_adata.obsm["spatial"], metric="euclidean"))
    )

    if annotation_model_map:
        rna_adata = annotate_cell_type_by_domain(
            rna_adata,
            annotation_key,
            annotation_model_map,
            cell_type_key="cell_type",
        )

    if use_cache:
        preprocess_path.mkdir(parents=True, exist_ok=True)
        rna_adata.write_h5ad(cache_path)
    return rna_adata


rna_adata = load_mouse_embryo_inline(
    raw_path,
    preprocess_path,
    use_cache=USE_CACHE,
    annotation_value=ORGAN,
    annotation_model_map=ANNOTATION_MODEL_MAP,
)

print(f"RNA shape:        {rna_adata.shape}")
print(f"RNA feat:         {rna_adata.obsm['feat'].shape}")
print(f"Ligand entries:   {len(ligand_receptor_map)}")
print(f"Cell types:       {rna_adata.obs['cell_type'].nunique() if 'cell_type' in rna_adata.obs else 'missing'}")


## Step 4 — Run CellSTIC

In [ ]:
result = run_cellstic(
    modality_datas=[rna_adata],
    ligand_receptor_map=ligand_receptor_map,
    model_path=model_path,
    output_path=result_path,
    config=cfg,
    device=device,
    auto_n_clusters=7,
)
print(f"AnnData saved to {result.adata_path}")

## Step 5 — Initialize Downstream Analysis

In [ ]:
adata = ad.read_h5ad(result_path / "cellstic_result.h5ad")

if ORGAN == "Brain":
    cell_type_filter = [
        "Blood",
        "Immune",
        "Gastrulation",
        "Vascular",
        "Neuron",
        "Ectoderm",
        "Fibroblast",
        "Neuroblast:",
        "Glioblast",
        "Radial glia",
    ]
else:
    cell_type_filter = None

analyzer = SingleLevelAnalysis.from_adata(
    adata,
    output_path=analysis_path,
    cell_type_key="cell_type",
    threshold=0.5,
    lr_filter=["F2:F2r", "Lpar3:Adgre5", "Nts:Sort1", "Plg:Pard3", "Thbs4:Cd36"],
    annotation_filter=["Brain", "Liver"],
    cell_type_filter=cell_type_filter,
)
print("SingleLevelAnalysis ready.")


## Step 6 — Cell-type Communication Heatmaps

In [ ]:
analyzer.run_cell_type_heatmaps()
display(SVG(filename=str(analysis_path / "cell_type_heatmaps" / "cell_type_pair_lr.svg")))


## Step 7 — Strength vs Spatial Distance

In [ ]:
analyzer.run_strength_vs_distance()
display(SVG(filename=str(analysis_path / "strength_vs_distance" / "strength_vs_distance.svg")))


## Step 8 — Spatial Communication Heatmaps

In [ ]:
analyzer.run_simple_heatmaps(font_size=18)
display_svg_grid(sorted((analysis_path / "spatial_heatmaps").glob("*.svg")))
print(f"Single-stage workflow completed. Results saved under: {analysis_path}")
